# VERITAS — live hallucination-technique benchmark on a free Colab GPU

Runs the full technique comparison (Baseline RAG, VERITAS, Semantic Entropy, Quote Grounding, Multi-Agent Consensus, Neurosymbolic Guardrails, Calibrated Selective Prediction, Graph-RAG) against a **real open-weights instruct model** — no API keys, no credits.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Default model is `Qwen/Qwen2.5-7B-Instruct` (Apache-2.0, ungated) 4-bit quantized so it fits a free T4 (~15 GB). No Hugging Face login required for Qwen; for gated models (Llama) log in in the optional cell.

In [ ]:
# 1. Confirm a GPU is attached
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Clone the repo and install (torch is already present on Colab)
!git clone -b claude/hallucination-reduction-rag-8whr4y https://github.com/aabhimittal/hallucination.git
%cd hallucination
!pip install -q -e . transformers accelerate bitsandbytes pandas

In [ ]:
# 3. (Optional) log in to Hugging Face — only needed for gated models like Llama.
# Skip this cell for the default Qwen model.
# from huggingface_hub import login; login()

In [ ]:
# 4. Run the live comparison. --balanced 5 = 5 of each question type (~15 Qs,
#    a few minutes). Drop --balanced for the full 40-question benchmark.
#    Swap --model for any instruct model, e.g. meta-llama/Llama-3.1-8B-Instruct.
!python benchmarks/run_comparison.py \
    --provider local \
    --model Qwen/Qwen2.5-7B-Instruct \
    --load-4bit \
    --balanced 5 \
    --out comparison_live

In [ ]:
# 5. Show the results table
from IPython.display import Markdown, display
display(Markdown(open('benchmarks/comparison_live.md').read()))

In [ ]:
# 6. (Optional) DoLa white-box comparison on the same GPU — vanilla vs
#    layer-contrasting decoding on a base model.
!python benchmarks/run_dola.py --model gpt2 --dola-layers high --limit 8
from IPython.display import Markdown, display
display(Markdown(open('benchmarks/dola.md').read()))

In [ ]:
# 7. (Optional) launch the interactive Gradio demo, driven by the local model,
#    with a public share link.
import os
os.environ['VERITAS_SHARE'] = '1'
# The demo defaults to the offline mock; use the CLI above for local-model
# numbers, or adapt app.py's build_llm to add a 'local' provider if you want
# the local model wired into the UI.
!python app.py